
## Training AI agent Decision Helper models 


In [18]:

## !pip install scikit-learn


In [19]:

# ================================
# CREWAI + TRAINED TOOL SELECTION
# ================================

from crewai import Agent
from langchain.tools import Tool

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



In [20]:

import os
from crewai_tools import SerperDevTool, BrowserbaseLoadTool, EXASearchTool
from crewai import Agent
from crewai import Task
from crewai import LLM

from langchain_community.llms import Ollama


from crewai import Crew, Process


In [21]:

# -------------------------------
# 1. DEFINE TOOLS
# -------------------------------
def calculator_tool(x: str):
    try:
        return str(eval(x))
    except:
        return "Error in calculation"


In [23]:



def search_tool(x: str):
    return f"Searching for: {x}"


In [24]:

# -------------------------------
# 2. LOGGING WRAPPER
# -------------------------------
logs = []

def logging_tool(tool_name, func):
    
    def wrapper(x):
        logs.append((x, tool_name))
        return func(x)
        
    return wrapper



In [25]:

tools = [
    Tool(
        name="Calculator",
        func=logging_tool("Calculator", calculator_tool),
        description="Use for math operations"
    ),
    Tool(
        name="Search",
        func=logging_tool("Search", search_tool),
        description="Use for general knowledge"
    )
]



In [26]:

# -------------------------------
# 3. CREWAI AGENT (FOR DATA COLLECTION)
# -------------------------------



agent = Agent(
    role='Assistant',
    goal='Solve problems using the correct tool',
    backstory="""You choose tools wisely.""",
    verbose=False,
    tools=tools,
    ## llm=llm_llama
    llm=LLM( model="ollama/llama3.2", base_url="http://localhost:11434" )
)


In [28]:

research_task = Task(
    description="Conduct comprehensive research on the latest fusion energy developments.",
    expected_output="A summary report highlighting key advancements in fusion energy that is really funny, with lots of dad jokes",
    agent=agent,
    ## tools=[ WebsiteSearchTool() ],
    async_execution=True
)


In [29]:

my_crew = Crew(
    agents=[agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)


In [30]:

result = my_crew.kickoff()
print(result)


# Agent: Assistant
## Task: Conduct comprehensive research on the latest fusion energy developments.


# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"fusion energy latest developments\"}"
## Tool Output: 
Searching for: fusion energy latest developments


# Agent: Assistant
## Final Answer: 
Welcome to the "Fusion Frenzy" report, folks! Buckle up and get ready for some seriously hot updates on the latest fusion energy developments. Because, let's face it, when it comes to nuclear fusion, we're talking about a reaction that's hotter than my Aunt Mildred's temper.

First off, we've got "Project ITER," which is like the ultimate fusion experiment gone wrong... right? Just kidding, it's going amazingly well! This European Organization for Nuclear Research (CERN) project aims to achieve controlled nuclear fusion by heating up plasma to temperatures of around 150 million degrees Celsius. That's hotter than a freshly baked pizza in a wood-fired oven!

But wait, there's mo

In [31]:

print(f"Raw Output: {result.raw}")


Raw Output: Welcome to the "Fusion Frenzy" report, folks! Buckle up and get ready for some seriously hot updates on the latest fusion energy developments. Because, let's face it, when it comes to nuclear fusion, we're talking about a reaction that's hotter than my Aunt Mildred's temper.

First off, we've got "Project ITER," which is like the ultimate fusion experiment gone wrong... right? Just kidding, it's going amazingly well! This European Organization for Nuclear Research (CERN) project aims to achieve controlled nuclear fusion by heating up plasma to temperatures of around 150 million degrees Celsius. That's hotter than a freshly baked pizza in a wood-fired oven!

But wait, there's more! The United States has its own fusion game on with the "National Ignition Facility" (NIF). This place is like the ultimate laser tag arena, where scientists are blasting tiny pellets of hydrogen fuel at each other to achieve fusion. It's like a high-energy game of dodgeball, but instead of balls, t

In [32]:

import pandas as pd
df = pd.read_csv("sales.csv")


In [33]:


df


,Date,Revenue,Expenses,Customers
0,2024-01-31,12000,8000,234
1,2024-02-29,10000,8500,201
2,2024-03-31,9300,8700,198
3,2024-04-30,11500,8100,225
4,2024-05-31,9800,7900,210
5,2024-06-30,10400,8000,215


In [34]:

import pandas as pd
from crewai import Agent, Task, Crew

from langchain.tools import tool

# Load CSV
df = pd.read_csv("sales.csv")

@tool
def summarize_csv() -> str:
    """Summarizes the dataframe by showing the first few rows and statistical metrics."""
    summary = df.describe(include='all').to_string()
    preview = df.head().to_string()
    return f"Data Preview:\n{preview}\n\nSummary Statistics:\n{summary}"



analyst = Agent(
    role='CSV Analyst',
    goal='Read the spreadsheet and summarize key metrics',
    backstory="""You're a researcher at a large company.
    You're responsible for analyzing data and providing insights
    to the business.""",
    verbose=False,
    ## llm=llm_llama
    tools=[   summarize_csv  ],
    llm=LLM( model="ollama/llama3.2", base_url="http://localhost:11434" )
)



task = Task(
    description="Summarize the spreadsheet, including any trends or patterns.",
    expected_output="A markdown-style summary report of the dataset including basic statistics and key insights.",
    agent=analyst
)

# === Crew ===
crew = Crew(agents=[analyst], tasks=[task], verbose=True)

if __name__ == "__main__":
    result = crew.kickoff()
    print("\n--- Summary Report ---\n")
    print(result)


2026-04-15 17:19:07,501 - 8663405120 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: CSV Analyst
## Task: Summarize the spreadsheet, including any trends or patterns.


# Agent: CSV Analyst
## Thought: Thought: I need to summarize the spreadsheet by reading its contents.
## Using tool: summarize_csv
## Tool Input: 
"{\"df\": \"spreadsheet_data.csv\"}"
## Tool Output: 
Data Preview:
         Date  Revenue  Expenses  Customers
0  2024-01-31    12000      8000        234
1  2024-02-29    10000      8500        201
2  2024-03-31     9300      8700        198
3  2024-04-30    11500      8100        225
4  2024-05-31     9800      7900        210

Summary Statistics:
              Date       Revenue    Expenses   Customers
count            6      6.000000     6.00000    6.000000
unique           6           NaN         NaN         NaN
top     2024-01-31           NaN         NaN         NaN
freq             1           NaN         NaN         NaN
mean           NaN  10500.000000  8200.00000  213.833333
std            NaN   1043.072385   322.49031   13.876839
min    

In [35]:

# -------------------------------
# 4. GENERATE DATA (RUN AGENT)
# -------------------------------
queries = [
    "2 + 2",
    "10 * 5",
    "100 / 4",
    "Who is Einstein?",
    "capital of France",
    "latest news",
    "3 * 9",
    "square root of 16"
]


In [36]:


print("\n--- Collecting data from agent ---\n")



--- Collecting data from agent ---



In [45]:

for q in queries:
    try:
        agent.execute_task(q)
    except:
        pass

print("Collected logs:")
print(logs)

logs.append(('2+2', 'Calculator'))

print(logs)


Collected logs:
[('fusion energy latest developments', 'Search')]
[('fusion energy latest developments', 'Search'), ('2+2', 'Calculator')]


In [46]:


# -------------------------------
# 5. TRAIN MODEL (QUERY -> TOOL)
# -------------------------------
texts = [x[0] for x in logs]
labels = [x[1] for x in logs]


In [47]:

texts


['fusion energy latest developments', '2+2']

In [48]:

labels


['Search', 'Calculator']

In [49]:

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

dtm_pd = pd.DataFrame( X.toarray(), index=texts, columns=vectorizer.get_feature_names_out()  ).head(10)
dtm_pd


,developments,energy,fusion,latest
fusion energy latest developments,0.5,0.5,0.5,0.5
2+2,0.0,0.0,0.0,0.0


In [50]:



model = LogisticRegression()
model.fit(X, labels)

print("\n--- Model trained ---\n")



--- Model trained ---



In [51]:

# -------------------------------
# 6. TRAINED POLICY (PREDICT TOOL)
# -------------------------------
def predict_tool(query):
    X = vectorizer.transform([query])
    return model.predict(X)[0]


In [52]:

# -------------------------------
# 7. FINAL AGENT (USES TRAINED POLICY)
# -------------------------------
def run_agent(query):
    chosen_tool = predict_tool(query)

    print(f"\nQuery: {query}")
    print(f"Chosen Tool (trained): {chosen_tool}")

    if chosen_tool == "Calculator":
        result = calculator_tool(query)
    else:
        result = search_tool(query)

    print(f"Result: {result}")
    return result


In [53]:

# -------------------------------
# 8. TEST
# -------------------------------
print("\n--- Testing trained agent ---")

run_agent("5 * 6")
run_agent("Who invented electricity?")
run_agent("12 + 45")
run_agent("population of Japan")




--- Testing trained agent ---

Query: 5 * 6
Chosen Tool (trained): Calculator
Result: 30

Query: Who invented electricity?
Chosen Tool (trained): Calculator
Result: Error in calculation

Query: 12 + 45
Chosen Tool (trained): Calculator
Result: 57

Query: population of Japan
Chosen Tool (trained): Calculator
Result: Error in calculation


'Error in calculation'